In [2]:
!pip install datasets transformers==4.40.2 accelerate==0.29.3 evaluate==0.4.1 bitsandbytes==0.43.1 trl==0.8.6 peft==0.10.0


In [3]:
pip install --upgrade transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 62.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 80.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.40.2
    Uninstalling transformers-4.40.2:
      Successfully uninstalled transformers-4.40.2
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install flash-attn --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 46.3 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Created wheel for flash-attn: filename=flash_attn-2.7.4.post1-cp310-cp310-linux_x86_64.whl size=187797312 sha256=b267f80a08e516292cdd748056a2178a45b8abedf7fca123292eb17c21c8c87c
  Stored in directory: /root/.cache/pip/wheels/59/ce/d5/08ea07bfc16ba218dc65a3a7ef9b6a270530bcbd2cea2ee1ca
Successfully built flash-attn
Note: you may need to restart the kernel to use updated packages.


In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from datasets import load_dataset, Dataset
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import logging
from dataclasses import dataclass, field
import os
import random
import torch
from datasets import load_dataset
from tqdm import tqdm
from trl import  TrlParser
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    HfArgumentParser,
    BitsAndBytesConfig,
        set_seed,

)
from trl import setup_chat_format
from peft import LoraConfig


from trl import (
   SFTTrainer)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [6]:

LLAMA_3_CHAT_TEMPLATE = (
    "{% for message in messages %}"
        "{% if message['role'] == 'system' %}"
            "{{ message['content'] }}"
        "{% elif message['role'] == 'user' %}"
            "{{ '\n\nHuman: ' + message['content'] +  eos_token }}"
        "{% elif message['role'] == 'assistant' %}"
            "{{ '\n\nAssistant: '  + message['content'] +  eos_token  }}"
        "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '\n\nAssistant: ' }}"
    "{% endif %}"
)

tqdm.pandas()

In [ ]:
DATASET_NAME = "codeparrot/apps"
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
my_token = ""
token = ""

In [8]:
train_data = load_dataset(DATASET_NAME, split="train")


README.md:   0%|          | 0.00/5.63k [00:00<?, ?B/s]

apps.py:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

The repository for codeparrot/apps contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/codeparrot/apps.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


train.jsonl:   0%|          | 0.00/107M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/1.29G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [9]:
data_list = list(train_data)
train_data = Dataset.from_list(data_list)
train_df = train_data.to_pandas()
train_df.head()

,problem_id,question,solutions,input_output,difficulty,url,starter_code
0,0,Polycarp has $n$ different binary words. A wor...,"[""for _ in range(int(input())):\n n = int(i...","{\n ""inputs"": [\n ""4\n4\n0001\n1000\n0011\...",interview,https://codeforces.com/problemset/problem/1259/D,
1,1,Mikhail walks on a Cartesian plane. He starts ...,"[""q=int(input())\n\nfor e in range(q):\n x,...","{\n ""inputs"": [\n ""3\n2 2 3\n4 3 7\n10 1 9...",interview,https://codeforces.com/problemset/problem/1036/B,
2,2,"You are given three sequences: $a_1, a_2, \ldo...","[""import sys\nimport random\nfrom fractions im...","{\n ""inputs"": [\n ""5\n3\n1 1 1\n2 2 2\n3 3...",interview,https://codeforces.com/problemset/problem/1408/A,
3,3,"You have $n$ barrels lined up in a row, number...","[""def solve():\n n, k = map(int,input().spl...","{\n ""inputs"": [\n ""2\n4 1\n5 5 5 5\n3 2\n0...",interview,https://codeforces.com/problemset/problem/1430/B,
4,4,"You are given a permutation $p=[p_1, p_2, \ldo...","[""for _ in range(int(input())):\n input()\n...","{\n ""inputs"": [\n ""3\n6\n4 5 1 3 2 6\n5\n5...",interview,https://codeforces.com/problemset/problem/1265/B,


In [ ]:
clean_df = train_df[["question","solutions"]]
clean_df.head()

,question,solutions
0,Polycarp has $n$ different binary words. A wor...,"[""for _ in range(int(input())):\n n = int(i..."
1,Mikhail walks on a Cartesian plane. He starts ...,"[""q=int(input())\n\nfor e in range(q):\n x,..."
2,"You are given three sequences: $a_1, a_2, \ldo...","[""import sys\nimport random\nfrom fractions im..."
3,"You have $n$ barrels lined up in a row, number...","[""def solve():\n n, k = map(int,input().spl..."
4,"You are given a permutation $p=[p_1, p_2, \ldo...","[""for _ in range(int(input())):\n input()\n..."


In [11]:
clean_df.to_csv('codeparrot-cleand.csv')

In [12]:
train_data = Dataset.from_pandas(clean_df)

In [13]:
system_message = """You are a LLM coding and problem solving assistant, answer the given problem in code, give the function to solve and the main driver program - together."""
def create_conversation(record):
    sample = {"messages": [
        
        {"role": "user", "content": f"""Give the python code that solves the problem: {record["question"]}"""},
        {"role" : "assistant", "content": f"""{record["solutions"]}"""}
    ]}
    return sample


train_data = train_data.map(create_conversation, batched=False)



#train_data.to_json("data/train_dataset.json", orient="records", force_ascii=False)


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [14]:
temp = train_data.to_pandas()
temp.head()

,question,solutions,messages
0,Polycarp has $n$ different binary words. A wor...,"[""for _ in range(int(input())):\n n = int(i...",[{'content': 'Give the python code that solves...
1,Mikhail walks on a Cartesian plane. He starts ...,"[""q=int(input())\n\nfor e in range(q):\n x,...",[{'content': 'Give the python code that solves...
2,"You are given three sequences: $a_1, a_2, \ldo...","[""import sys\nimport random\nfrom fractions im...",[{'content': 'Give the python code that solves...
3,"You have $n$ barrels lined up in a row, number...","[""def solve():\n n, k = map(int,input().spl...",[{'content': 'Give the python code that solves...
4,"You are given a permutation $p=[p_1, p_2, \ldo...","[""for _ in range(int(input())):\n input()\n...",[{'content': 'Give the python code that solves...


In [15]:
model_id = "meta-llama/Llama-3.2-3B-Instruct"
use_bf16 = True

In [16]:
max_seq_length=512

In [17]:
import logging
from dataclasses import dataclass, field
import os

import random
import torch
from datasets import load_dataset
from tqdm import tqdm
from trl.commands.cli_utils import  TrlParser
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    HfArgumentParser,
    BitsAndBytesConfig,
        set_seed,

)
from trl import setup_chat_format
from peft import LoraConfig


from trl import (
   SFTTrainer)


tqdm.pandas()

# @dataclass
# class ScriptArguments:
#     dataset_path: str = field(
#         default=None,
#         metadata={
#             "help": "Path to the dataset"
#         },
#     )
#     model_id: str = field(
#         default=None, metadata={"help": "Model ID to use for SFT training"}
#     )
#     max_seq_length: int = field(
#         default=512, metadata={"help": "The maximum sequence length for SFT Trainer"}
#     )
#     use_qlora: bool = field(default=False, metadata={"help": "Whether to use QLORA"})
#     merge_adapters: bool = field(
#         metadata={"help": "Whether to merge weights for LoRA."},
#         default=False,
#     )



In [18]:
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True,token=token)
tokenizer.pad_token = tokenizer.eos_token
#tokenizer.chat_template = LLAMA_3_CHAT_TEMPLATEtorch.backends.cuda.matmul.allow_tf32 = True


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [61]:
torch.backends.cuda.matmul.allow_tf32 = True

In [19]:
def template_dataset(examples):
    return{"messages":  tokenizer.apply_chat_template(examples["messages"], tokenize=False)}
    
train_data = train_data.map(template_dataset, remove_columns=["messages"])


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [20]:
updated_train = train_data.to_pandas()
updated_train.head()


,question,solutions,messages
0,Polycarp has $n$ different binary words. A wor...,"[""for _ in range(int(input())):\n n = int(i...",<|begin_of_text|><|start_header_id|>system<|en...
1,Mikhail walks on a Cartesian plane. He starts ...,"[""q=int(input())\n\nfor e in range(q):\n x,...",<|begin_of_text|><|start_header_id|>system<|en...
2,"You are given three sequences: $a_1, a_2, \ldo...","[""import sys\nimport random\nfrom fractions im...",<|begin_of_text|><|start_header_id|>system<|en...
3,"You have $n$ barrels lined up in a row, number...","[""def solve():\n n, k = map(int,input().spl...",<|begin_of_text|><|start_header_id|>system<|en...
4,"You are given a permutation $p=[p_1, p_2, \ldo...","[""for _ in range(int(input())):\n input()\n...",<|begin_of_text|><|start_header_id|>system<|en...


In [51]:
torch_dtype = torch.float16 
#quant_storage_dtype = torch.float16


print(f"Using QLoRA - {torch_dtype}")
quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch_dtype,
            
    )

        


Using QLoRA - torch.float16


In [77]:
import transformers
training_args = transformers.TrainingArguments(
      per_device_train_batch_size=4,
      num_train_epochs=3,
      learning_rate=2e-4,
      fp16=False,
      bf16=False,
      output_dir = "/kaggle/working/",
      dataloader_num_workers=2,
      optim="paged_adamw_8bit",
      lr_scheduler_type="cosine",
      warmup_ratio=0.05,
      report_to="none",
      # use_dora=False,
      # use_rslora=False
      # lora_alpha=8,
      # lora_dropout=0.0
)


In [78]:

#training_args.optim = "adamw_bnb_8bit"  
training_args.gradient_checkpointing = True  # Recompute activations
training_args.gradient_accumulation_steps = 2  # Accumulate gradients to reduce VRAM load

In [79]:
training_args.per_device_train_batch_size = 8 # Adjust based on memory
training_args.per_device_eval_batch_size = 8
training_args.dataloader_pin_memory = True
training_args.dataloader_num_workers = 4 
#training_args.fsdp = None  #"full_shard auto_wrap"

In [80]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    #device_map="auto",
    device_map="auto",
    attn_implementation="sdpa", # use sdpa, alternatively use "flash_attention_2", "sdpa"
    #torch_dtype=torch.float16, # can't use bf16
    use_cache= True,  # this is needed for gradient checkpointing
    token=token
)


model.gradient_checkpointing_enable()

    ################
    # PEFT
    ################
    # LoRA config based on QLoRA paper & Sebastian Raschka experiment
peft_config = LoraConfig(
    lora_alpha=8,
    lora_dropout=0.05,
    r=8,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

    ################
    # Training
    ################
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    dataset_text_field="messages",
    peft_config=peft_config,
    max_seq_length=256,
    tokenizer=tokenizer,
    packing=True,
    dataset_kwargs={
        "add_special_tokens": False,  # We template with special tokens
        "append_concat_token": False,  # No need to add additional separator token
    },
)

trainer.model.print_trainable_parameters()

    ##########################
    # Train model
    ##########################
    # checkpoint = None
    # if training_args.resume_from_checkpoint is not None:
    #     checkpoint = training_args.resume_from_checkpoint
    # trainer.train(resume_from_checkpoint=checkpoint)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/trl/trainer/sft_trainer.py:323: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:469: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 2,293,760 || all params: 3,215,043,584 || trainable%: 0.07134460047182987


In [ ]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


In [ ]:
model.save_pretrained()

In [ ]:
model.push_to_hub("Rudrresh/llama3-codeparrot",token=my_token)
trainer.tokenizer.push_to_hub("Rudrresh/llama3-codeparrot",token=my_token)